# W2D1：进入DataFrame世界——pandas核心思维

**今天要解决的问题**：你手上有一份5千行的电商订单CSV，mentor小王要你"先看看数据长什么样，有没有问题"——你怎么办？
**你要用到的武器**：DataFrame初探六板斧 + 行列选取(loc/iloc) + 布尔索引 + 排序统计 + 列操作
**需要的前置知识**：W1的list/dict（容器）、for/if（控制流）、def（函数）、csv.DictReader（文件读取）

---

### 📋 今日知识地图

- [ ] **pandas世界观**：Series=带标签的list、DataFrame=dict of Series、pandas是W1手写代码的工业级实现
- [ ] **数据读入**：`pd.read_csv()`(encoding/nrows/usecols/dtype/parse_dates)、`read_excel()`、`read_json()`、路径/BOM/文件太大三个陷阱
- [ ] **初探六板斧**：`head()`/`tail()`/`info()`/`describe()`/`shape`/`columns.tolist()`/`dtypes`/`value_counts()`
- [ ] **行列操作**：`df['列']`/`df[['A','B']]`(双括号!)、`loc`(标签，含右边界)/`iloc`(位置，不含右边界)/`at`/`iat`
- [ ] **布尔索引**：单条件/`&`(AND)/`|`(OR)/`isin()`/`~`(取反)/`str.contains()`/组合条件
- [ ] **排序与统计**：`sort_values()`/`nlargest()`/`nsmallest()`/`unique()`/`nunique()`
- [ ] **列操作**：新增列(`df['新列']=`)/`assign()`/`drop()`/`rename()`

---

## 🔴 先别往下看——来试试这个

你在速卖通做初级数据分析师。mentor小王丢给你一份CSV文件，说：

> "这份订单数据有5000行，你先看看数据质量怎么样。我想知道：哪些城市卖得最好？有没有异常的大额订单？数据有没有缺失？5分钟后告诉我。"

你现在手上只有W1学的纯Python——list、dict、for循环、csv.DictReader。

**别往下翻。在脑子里想一下**：用W1的工具，你要写多少行代码才能回答小王的3个问题？

- 每个城市的销售额：遍历5000行→按城市分类→分别求和→排序……大概15-20行
- 异常大额订单：遍历→if判断→收集→排序……大概10行
- 数据缺失：遍历→检查每个字段→统计……大概10行

**总共大概要写40-50行Python，而且运行起来要好几秒。**

如果数据不是5000行，而是50万行——你W1写的for循环可能要跑几分钟。

你刚才感受到的这种"能做但太费劲"的感觉——今天要学的pandas就是专门解决它的。

用pandas，小王的3个问题：**8行代码，0.1秒出结果。**

---

## 概念1：pandas世界观 + 数据读入 + 初探六板斧

### 🤔 你现有的知识够用吗？

W1你学了list和dict。list是一排数据——你知道第0个是什么，但不知道"北京"对应的值是多少。dict能按标签取值，但它的每个value只能是一个值，不能是一列。

现在你有这样一份数据：

| 订单号 | 城市 | 金额 |
|--------|------|------|
| 1001 | 北京 | 350 |
| 1002 | 上海 | 280 |
| 1003 | 北京 | 520 |

用W1的工具：要存它→list套dict，每个dict是一行。要查"所有北京的订单"→遍历所有行→if判断→收集。要算"北京的总金额"→再遍历→累加。

每个操作都要写循环。而且list套dict这种结构——你不能直接"看整列"，不能直接"算整列"，不能直接"筛整列"。

你刚才撞到的这个墙，**pandas DataFrame** 就是专门解决它的。

### pandas世界观 + 数据读入 + 初探六板斧

**① 说白了就是**：W1你学的list是一排抽屉（按编号找东西），dict是贴了标签的抽屉（按标签找东西）。pandas的Series = 贴了标签的一排抽屉——既能按编号找，又能按标签找。pandas的DataFrame = 一张Excel表格——每一列是一个Series，整张表是一个"dict of Series"（列名→整列数据）。

**② 在沙盘里怎么用**：你在速卖通，mentor小王给你`orders.csv`（5000行、8列：order_id/user_id/product_id/amount/quantity/date/city/category）。你一行`pd.read_csv()`把整个文件读进一个DataFrame变量`df`。然后用六板斧——`df.head()`看前5行长什么样，`df.info()`看每列类型和缺失情况（这是最重要的一步！），`df.describe()`看金额的均值/最大最小值/分位数——3行代码，小王的"数据质量怎么样"基本就回答了。

**③ 跟你上周学的list/dict的关系**：DataFrame本质上就是W1的"list套dict"的工程化版本。W1你手写`[{'订单号':1001,'城市':'北京','金额':350}, {...}, ...]`存表格数据→pandas用一个DataFrame一行搞定，而且底层用C实现，比Python快100-500倍。W1的`csv.DictReader+for循环+list.append`→今天的`pd.read_csv()`一行替代。W1手写`sum()/len()/max()/min()+排序取中位数`→今天的`df.describe()`一行替代。

**④ 为什么这样设计**：pandas不是"一个全新的要背的库"——它是把W1你手写的那些"用list/dict管理表格数据"的代码，做成了工业级实现。Series底层是NumPy的ndarray（C语言的连续内存数组），DataFrame底层是多个ndarray的集合。所以pandas操作整列时→数据在C层面连续排列→CPU可以一次处理多个（SIMD指令）→比Python for循环快100-500倍。这就是为什么同样的逻辑，pandas一行比W1的15行快两个数量级。你是在写"数据分析"不是在写"如何遍历list"——pandas让你专注于分析逻辑本身。

In [ ]:
# 概念1落地代码：从读文件到初探，6行搞定
import pandas as pd

# 读入——W1的open()+csv.DictReader()+for+list.append() = 这一行
df = pd.read_csv('orders.csv', encoding='utf-8')

# 六板斧——6行代码回答"数据长什么样"
print(type(df))           # <class 'pandas.core.frame.DataFrame'>
print(df.shape)           # (5000, 8) → 5000行8列
print(df.head(3))         # 前3行——直观感受
print(df.info())          # 每列类型+非空数量+内存——最重要！
print(df.describe())      # 数值列的count/mean/std/min/25%/50%/75%/max
print(df.columns.tolist())# 所有列名的list——方便复制粘贴

In [ ]:
# 概念1即时小练习：用六板斧初探你的第一份数据
# 用pd.read_csv读入orders.csv（假设文件在notebook同目录下）
# 然后依次跑：head(10) / tail(5) / info() / describe() / shape / dtypes / columns.tolist()
# 每跑一条，在旁边用注释写一句"我发现了什么"
# 例如：describe()显示amount的75%分位=500但max=99999→有极端值！

# 你的代码：


---

## 概念2：行列选取 + 布尔索引 + 排序统计 + 列操作

### 🤔 你现有的知识够用吗？

初探完数据，mentor小王又来了：

> "帮我把'北京'且'金额>500'的订单筛出来，按金额从大到小排，只看订单号、金额、品类三列。再新增一列'税后金额'=金额×0.87。"

用W1的list套dict——筛选用for+if（4行）、排序用sorted+lambda（2行）、选列用list comprehension（2行）、新增列用for循环（3行）——大概11行。

而且有一个隐蔽的坑：W1里你要判断"城市是北京或上海"，写`if o['城市']=='北京' or o['城市']=='上海'`没问题。但在pandas里，如果你写了`df[df['城市']=='北京' or df['城市']=='上海']`——**直接报错！**

你刚才感受到的"同样的逻辑，但语法不一样，还容易踩坑"——今天概念2就是专门解决它的。

### 行列选取 + 布尔索引 + 排序统计 + 列操作

**① 说白了就是**：DataFrame像一张Excel表格。行列选取就是"我要第几行到第几行、第几列到第几列"——有两种说法：按名字选（loc，像"我要'张三'到'李四'这几行"）和按位置选（iloc，像"我要第1到第5行"）。布尔索引就是Excel的"筛选"功能——点一下"金额>500"的筛选按钮。排序就是Excel的"排序"。新增列就是Excel的"在右边加一列然后写公式"。

**② 在沙盘里怎么用**：小王的那个需求，pandas一行链式操作就搞定了：先用布尔索引筛出"北京+金额>500"的行（`df[(df['city']=='北京') & (df['amount']>500)]`），`.loc`选需要的列，`.sort_values('amount', ascending=False)`排序，`df['税后金额'] = df['amount'] * 0.87`新增列。关键语法：多条件必须用`&`（不是`and`！），每个条件必须用括号包起来！

**③ 跟你上周学的for/if的关系**：布尔索引`df[df['amount']>500]` = W1的`result=[]; for o in orders: if o['amount']>500: result.append(o)`——刚才那4行Python，现在1行pandas。`df.sort_values('amount')` = W1的`sorted(orders, key=lambda x:x['amount'])`。`df['amount'].unique()` = W1的`set(lst)`。`df['税后金额'] = df['amount']*0.87` = W1的`for o in orders: o['税后金额'] = o['amount']*0.87`。每一个pandas操作，底层都是W1你手写过的逻辑——只不过pandas把它做成了C语言实现，一行顶你十行。

**④ 为什么这样设计**：pandas的loc和iloc两套体系不是"故意搞复杂"——它们对应两种不同的思考方式。loc按标签选（"我要名叫'张三'的那行"）——适合你知道数据含义但不知道位置时。iloc按位置选（"我要第5行"）——适合你知道位置但标签可能是任意字符串时。loc[0:5]包含右边界（标签5）是因为标签选择时"包含"比"不包含"更直观——你说"从北京到深圳"自然包含深圳。iloc[0:5]不包含右边界是为了和Python的list切片`lst[0:5]`保持一致——这是Python的惯例。布尔索引为什么必须用`&`/`|`而不是`and`/`or`？因为`and`/`or`是Python级别的逻辑运算（操作单个True/False），而`&`/`|`是NumPy/pandas级别的按位运算（操作True/False数组——一整个列）。你是在对10000个True/False做"同时为真"的判断，不是对1个。

In [ ]:
# 概念2落地代码：选取+筛选+排序+新增，pandas一行的威力

# --- 行列选取 ---
df['amount']                     # 选一列→返回Series（像W1的dict['key']）
df[['order_id','amount','city']] # 选多列→双括号！！单括号df['A','B']报错！
df.loc[0:5, ['amount','city']]   # 按标签：0到5行（包含5！），amount和city列
df.iloc[0:5, 0:3]               # 按位置：0到4行（不含5！），前3列
df.at[3, 'amount']               # 单个值——比.loc快

# --- 布尔索引（SQL WHERE的pandas版）---
df[df['amount'] > 500]                                    # 单条件
df[(df['amount'] > 500) & (df['city'] == '北京')]          # AND——必须用&！必须括号！
df[(df['city'] == '北京') | (df['city'] == '上海')]        # OR——必须用|！
df[df['city'].isin(['北京','上海','深圳'])]                  # 多值→比多个OR简洁
df[~df['city'].isin(['北京','上海'])]                       # 取反——~是NOT
df[df['category'].str.contains('耳机', case=False)]        # 字符串筛选

# --- 排序 + 统计 + 列操作 ---
df.sort_values('amount', ascending=False)                  # 金额降序
df.sort_values(['city','amount'], ascending=[True, False]) # 先城市升序，再金额降序
df['amount'].nlargest(10)          # 最大10笔——比sort+head快
df['city'].value_counts()          # 每个城市出现多少次
df['city'].value_counts(normalize=True)  # 变成比例，比绝对数直观
df['city'].nunique()               # 有多少不同城市=len(unique())
df['税后金额'] = df['amount'] * 0.87          # 新增列（W1: for循环）
df = df.assign(利润=df['amount']-100, 利润率=lambda x: x['利润']/df['amount'])  # 一次新增多列
df.drop(['temp_col'], axis=1)       # 删列；axis=1列/axis=0行
df.rename(columns={'amount':'金额'}) # 重命名列

In [ ]:
# 概念2即时小练习：loc vs iloc + 布尔索引
# 1. 分别用loc和iloc取df的前5行、第10-15行+金额和城市两列、第3行金额那个单元格
#    在注释中标注：哪个参数是标签、哪个是位置
# 2. 布尔索引四连：①金额>500的行 ②北京+金额>300的行 ③北京或上海的行 ④品类含"电子"的行
#    每步后打印len()看筛选出多少行

# 你的代码：


In [ ]:
## 📝 基础练习1：六板斧完整探索
## 场景：速卖通电商 | 用到的武器：read_csv + 六板斧（新）

# 读取orders.csv，完整跑一遍六板斧：
# head(10) / tail(5) / info() / describe() / shape / dtypes / columns.tolist()
# 每条结果后面用print()输出一句"我发现了什么"
# 例如：print(f"发现：describe显示amount的75%分位={df['amount'].quantile(0.75):.0f}但max={df['amount'].max():.0f}→有极端值")
# 至少写5条发现，包括：数据规模、缺失情况、类型问题、异常值、分布特征

# 你的代码：


In [ ]:
## 📝 基础练习2：value_counts + 排序 + 列操作
## 场景：速卖通电商 | 用到的武器：value_counts + sort_values + assign（新）

# 1. 用value_counts看城市分布和品类分布——哪个城市订单最多？哪个品类销量最高？
# 2. 用sort_values找金额最大的10笔订单——输出订单号/城市/品类/金额
# 3. 新增一列"税后金额"=amount*0.87，再新增一列"单价"=amount/quantity
# 4. 用nlargest(10)找单价最高的10笔——和金额Top10有没有重合？

# 你的代码：


In [ ]:
## 🎮 进阶练习1：手游玩家表初探（换场景！）
## 场景：手游工作室 | 用到的武器：六板斧+value_counts+describe+dtypes（新）
## 覆盖知识点：初探六板斧、dtypes发现类型问题、value_counts看分布、describe发现极值

# 假设你拿到一份手游玩家数据 players.csv（列：user_id/充值金额/等级/服务器ID/注册日期/最后登录）
# 1. 读入后用六板斧完整探索
# 2. 重点检查：dtypes——"注册日期"和"最后登录"是不是object？（应该是日期！）
# 3. value_counts看等级分布——玩家集中在哪个等级段？
# 4. describe看充值金额——分布均匀吗？（看mean vs 50%差距、max vs 75%差距）
# 5. 写5条发现：数据规模、类型问题、等级分布、充值分布（是否极不均匀）、缺失情况

# 你的代码：


In [ ]:
## 💰 进阶练习2：金融交易表探索（换场景！）
## 场景：理财平台 | 用到的武器：value_counts(normalize)+nlargest+loc/iloc（新）+ W1函数（旧）
## 覆盖知识点：normalize看比例、nlargest找极值、loc/iloc取子集、对比两种选取方式

# 假设你拿到一份理财交易数据 transactions.csv
# （列：user_id/product_id/trans_type/amount/balance_after/trans_date/channel）
# 1. value_counts(normalize=True)看trans_type的分布比例——哪种交易类型最多？
# 2. nlargest(10)找金额最大的10笔交易
# 3. 分别用loc和iloc取：前20行+amount和trans_type两列、第50-60行全部列
#    对比两种写法的差异——哪个用标签？哪个用位置？
# 4. 定义函数describe_column(df, col)，返回该列的样本量/缺失率/均值/中位数/标准差/min/max/IQR
#    （用W1学的def——今天在pandas场景下复用）
#    然后用这个函数分析amount列

# 你的代码：


In [ ]:
## 🏆 挑战练习1：三份无数据字典的CSV——独立初探
## 场景：速卖通电商 | 用到的武器：全部新概念 + W1全部基础
## 覆盖知识点：六板斧全流程、数据质量问题识别、业务推理

# 假设mentor小王丢给你三份CSV（orders.csv / users.csv / products.csv），没有任何说明文档。
# 你的任务：分别初探三份文件，每份写100字以上的描述，包括：
#   - 数据规模（多少行多少列）
#   - 列含义猜测（根据列名推断每列代表什么）
#   - 数据质量问题（缺失值在哪里？哪些列类型不对？有没有异常值？）
#   - 三份文件之间的关系猜测（user_id可能关联用户表？product_id关联商品表？）
#   - 接下来需要了解什么才能继续分析？
#
# 最后用一段话总结：三份文件中，哪份数据质量最差？为什么？
#
# ⚠️ 至少试15分钟。如果卡住了，在代码注释里写下你卡在哪一步、你的思路是什么。

# 你的代码：


In [ ]:
## 🏆 挑战练习2：布尔索引综合——多条件筛选链
## 场景：速卖通电商 | 用到的武器：布尔索引全套（新）+ 排序（新）
## 覆盖知识点：多条件组合筛选、str.contains、~取反、分步验证

# 找出满足以下所有条件的订单：
#   条件1：金额>500
#   条件2：城市是北京或上海
#   条件3：品类名包含"电子"
#   条件4：品类名不包含"二手"（排除二手电子）
#
# 要求：每加一个筛选条件后，print(f"条件X后剩余{len(df_filtered)}行")验证行数
# 最后按金额降序排列，输出前20条结果（只显示order_id/city/category/amount四列）
#
# 关键检查：①每个&的条件都用括号包起来了吗？②用的是&不是and吗？
#          ③~的位置对吗？④str.contains的参数写对了吗？

# 你的代码：


In [ ]:
## 🚗 迁移练习：滴滴出行——全新领域，自己选武器
## 场景：滴滴出行 | 用到的武器：今天学的全部（你自己判断选哪个）
## 覆盖知识点：全量——六板斧/行列选取/布尔索引/排序统计/列操作

# 你拿到一份滴滴出行数据 trips.csv（列：trip_id/user_id/driver_id/city/
# distance_km/duration_min/fare/start_time/end_time）
#
# 这份数据你从来没见过，没有人告诉你"该用什么方法"。
# 你的任务：
#   1. 自己判断用今天学的哪些武器来探索这份数据
#   2. 写至少200字："我发现了什么" + "数据质量有什么问题" + "接下来需要了解什么"
#   3. 找出：最长的一次行程（距离和时间）、收入最高的10次行程（fare）、
#      哪个城市的行程最多、有没有距离为0或异常大的行程
#
# 题干到此为止。你需要自己判断今天学的哪个武器能解决这个问题。

# 你的代码和发现：


---

## ⚠️ 新手最容易踩的坑

### 坑1：Windows路径反斜杠`\`被当成转义符

❌ 错误写法：
```python
df = pd.read_csv('C:\data\new\orders.csv')  # \n被当成换行！\d被当成转义！
```
✅ 正确写法（三选一）：
```python
df = pd.read_csv('C:/data/new/orders.csv')      # 用正斜杠，最简单
df = pd.read_csv('C:\\data\\new\\orders.csv')   # 双反斜杠转义
df = pd.read_csv(r'C:\data\new\orders.csv')    # 前面加r——raw string
```
**为什么错**：Python字符串里`\n`是换行、`\t`是Tab、`\d`没有定义。路径里的`\`被当成转义字符的开头。

### 坑2：CSV有BOM头→第一列名变成乱码`ï»¿`

❌ 错误现象：`df.columns[0]`显示`ï»¿order_id`而不是`order_id`

✅ 正确写法：
```python
df = pd.read_csv('orders.csv', encoding='utf-8-sig')  # -sig去掉BOM
```
**为什么错**：某些Windows软件（如Excel）保存的CSV开头有BOM(Byte Order Mark)标记字节，`utf-8`编码不会去掉它，`utf-8-sig`会。

### 坑3：CSV文件太大→一次性读内存爆了

✅ 防御措施：
```python
df = pd.read_csv('big_file.csv', nrows=100)  # 先读100行看看结构
# 确认列名和类型后再决定怎样读全量
# 周四会学chunksize分块读——今天先用nrows探索
```

### 坑4：`loc`按标签 vs `iloc`按位置——搞混=数据全错

❌ 易错场景：
```python
# 你以为自己在按位置选→结果pandas在按标签选
df.loc[0:5]    # 按标签！如果标签0-5恰好是数字→返回6行（包含标签5）
df.iloc[0:5]   # 按位置！永远是前5行（位置0-4，不含5）
# 如果标签是字符串如'北京'/'上海'→loc['北京':'上海']包含上海
# iloc只接受数字位置→iloc['北京']直接报错！
```
✅ 记忆口诀：**loc = label（标签），iloc = index location（位置序号）。这是两个完全不同的坐标系——选之前先想清楚你用的是哪个。**

### 坑5：`loc[0:5]`包含标签5，但`iloc[0:5]`不包含位置5——同样的`0:5`行为不同！

❌ 最易出错的场景：
```python
df.loc[0:5]     # 返回6行（标签0到标签5→包含标签5！）
df.iloc[0:5]    # 返回5行（位置0到位置4→不含位置5！和Python list切片一样）
```
✅ **为什么会这样设计**：loc按标签选——"从北京到深圳"自然包含深圳→包含右边界更直观。iloc按位置选→为了和W1的`lst[0:5]`（不含5）保持一致→这是Python的切片惯例。**用之前想清楚：你现在用的是loc还是iloc？**

### 坑6：选多列必须双括号——单括号报KeyError

❌ 错误写法：
```python
df['order_id', 'amount']  # KeyError! pandas以为'order_id', 'amount'是单个key
```
✅ 正确写法：
```python
df[['order_id', 'amount']]  # 里面是一个list——告诉pandas"我要这两列"
```
**为什么错**：`df['A','B']`的括号里`'A','B'`被Python当成一个tuple `('A','B')`→pandas去列名里找这个tuple→找不到→KeyError。

### 坑7：多条件筛选必须用`&`/`|`并且每个条件加括号

❌ 错误写法1——用了Python的`and`：
```python
df[df['amount'] > 500 and df['city'] == '北京']  # 报错！
```
❌ 错误写法2——忘了括号：
```python
df[df['amount'] > 500 & df['city'] == '北京']  # 先算500 & df['city']→乱七八糟
```
✅ 正确写法：
```python
df[(df['amount'] > 500) & (df['city'] == '北京')]  # 每个条件一对括号，用&连接
```
**为什么错**：`and`/`or`是Python级别的逻辑运算（一次只能判断一个True/False），`&`/`|`是NumPy/pandas级别的按位运算（一次处理一整个True/False数组——整列）。括号是因为`&`的优先级比`>`高——不加括号会先算`500 & df['city']`再比较。

### 坑8：`df.info()`显示non-null数量——第一眼就该看这个

```python
df.info()
# <class 'pandas.core.frame.DataFrame'>
# RangeIndex: 5000 entries, 0 to 4999
# Data columns (total 8 columns):
#  #   Column      Non-Null Count  Dtype  
# ---  ------      --------------  -----  
#  0   order_id    5000 non-null   int64  
#  1   amount      5000 non-null   float64
#  2   city        4872 non-null   object  ← 只有4872！少了128行→有缺失！
#  3   category    5000 non-null   object  ← 类型是object→可能是字符串，检查一下
```
**记忆**：`info()`是你拿到任何新数据后必须跑的第一条命令。没有之一。Non-Null Count列是"缺失值探测器"——哪列的数字小于总行数，哪列就有缺失。

---

## 参考答案

### 概念1即时小练习 参考
```python
import pandas as pd
df = pd.read_csv('orders.csv', encoding='utf-8')
print(df.head(10))   # 发现：order_id从1001开始，city列有中文
print(df.tail(5))    # 发现：最后5行日期是6月30日，数据完整
print(df.info())     # 发现：city列只有4872 non-null→有128个缺失！category是object
print(df.describe()) # 发现：amount的max=99999是75%分位500的200倍→有极端值
print(df.shape)      # 发现：(5000, 8)→5000行8列
print(df.dtypes)     # 发现：date是object不是datetime→需要转换
print(df.columns.tolist())  # 发现：['order_id','user_id','product_id','amount','quantity','date','city','category']
```

### 概念2即时小练习 参考
```python
# 1. loc vs iloc
df.loc[0:4]          # 标签0-4，包含4→5行
df.iloc[0:5]          # 位置0-4，不含5→5行（和loc一样但语义不同！）
df.loc[10:15, ['amount','city']]   # 标签10-15行，amount和city列
df.iloc[10:16, [3,6]]              # 位置10-15行，第4和第7列
df.loc[3, 'amount']   # 标签为3那行的amount
df.iloc[3, 3]          # 位置第4行第4列

# 2. 布尔索引四连
print(len(df[df['amount'] > 500]))                          # ①
print(len(df[(df['amount'] > 300) & (df['city'] == '北京')])) # ② AND+括号
print(len(df[(df['city'] == '北京') | (df['city'] == '上海')])) # ③ OR+括号
print(len(df[df['category'].str.contains('电子', case=False)]))# ④ 字符串筛选
```

### 基础练习1 参考结构
```python
df = pd.read_csv('orders.csv', encoding='utf-8')
print(df.head(10))
print(f"发现：前10行中city列包含北京/上海/深圳，amount范围100-800")
print(df.info())
print(f"发现：总行数5000，city列non-null=4872→有128个缺失值！")
print(df.describe())
print(f"发现：amount均值={df['amount'].mean():.0f}，中位数={df['amount'].median():.0f}，max={df['amount'].max():.0f}→max是75%分位的200倍→存在极端大额订单")
# ... (继续head/tail/shape/dtypes/columns.tolist())
```

### 基础练习2 参考
```python
print(df['city'].value_counts())        # 北京订单最多？
print(df['category'].value_counts())    # 哪个品类第一？
print(df.sort_values('amount', ascending=False)[['order_id','city','category','amount']].head(10))
df['税后金额'] = df['amount'] * 0.87
df['单价'] = df['amount'] / df['quantity']
print(df.nlargest(10, '单价')[['order_id','city','category','amount','单价']])
# 看和amount Top10的重合度
```

### 进阶练习1 参考结构
```python
players = pd.read_csv('players.csv', encoding='utf-8')
print(players.head())
print(players.info())      # 发现：注册日期和最后登录是object！
print(players.describe())  # 发现：充值金额mean远大于50%→少数玩家充了大量钱
print(players['等级'].value_counts().sort_index())  # 发现：集中在10-30级
# 写5条发现...
```

### 进阶练习2 参考
```python
tx = pd.read_csv('transactions.csv', encoding='utf-8')
print(tx['trans_type'].value_counts(normalize=True))  # 比例分布
print(tx.nlargest(10, 'amount'))
print(tx.loc[:19, ['amount','trans_type']])   # 标签0-19=20行
print(tx.iloc[:20, [2,3]])                    # 位置0-19=20行

def describe_column(df, col):
    s = df[col].dropna()
    return {'列名':col, '样本量':len(s), '缺失率':f"{df[col].isna().mean():.1%}",
            '均值':round(s.mean(),2), '中位数':round(s.median(),2),
            '标准差':round(s.std(),2), '最小值':s.min(), '最大值':s.max(),
            'IQR':s.quantile(0.75)-s.quantile(0.25)}
print(describe_column(tx, 'amount'))
```

### 挑战练习1 评分要点
- 每份文件独立输出，不互相抄袭
- "列含义猜测"有推理过程（不只看列名，还看value_counts结果）
- "数据质量"至少覆盖：缺失值/类型问题/异常值
- "关系猜测"基于列名（user_id/product_id的关联）
- 最后一段总结有明确的"最差"判断+理由

### 挑战练习2 参考
```python
result = df[df['amount'] > 500]
print(f"条件1后: {len(result)}行")
result = result[result['city'].isin(['北京','上海'])]
print(f"条件2后: {len(result)}行")
result = result[result['category'].str.contains('电子', case=False)]
print(f"条件3后: {len(result)}行")
result = result[~result['category'].str.contains('二手', case=False)]
print(f"条件4后: {len(result)}行")
print(result.sort_values('amount', ascending=False)[['order_id','city','category','amount']].head(20))
```

### 迁移练习 评分要点
- 用六板斧做了初探（head/info/describe）
- 发现了fare的分布特征和异常值
- 用value_counts看了city分布
- 用nlargest找了最长/最贵行程
- 写了200字发现+质量问题+下一步
- 没有使用今天没教过的函数/方法

---

## 偏差诊断

| 如果你的输出/报错是... | 你可能的误解是... | 补救 |
|----------------------|-----------------|------|
| `KeyError: 'order_id'` 或列名找不到 | 列名记错了/大小写不对/有空格 | `print(df.columns.tolist())`复制粘贴列名，不要手打 |
| `KeyError: ('order_id', 'amount')` | 选了多列但用了单括号`df['A','B']` | 回到Cell 4b重看"行列选取"→必须双括号`df[['A','B']]` |
| `ValueError: The truth value of a Series is ambiguous` | 多条件筛选用`and`/`or`而不是`&`/`|` | 回到Cell traps坑6→把`and`换成`&`，`or`换成`|`，每个条件加括号 |
| `SyntaxError`指向`&`那一行 | 条件没加括号→运算符优先级错误 | 每个条件用括号包起来：`(df['A']>0) & (df['B']<10)` |
| `FileNotFoundError: orders.csv` | 路径不对或文件不在当前目录 | 检查文件路径→用`/`不用`\`→或者把文件放到notebook同目录 |
| describe()输出里看不到某些列 | describe()默认只看数值列→字符串列被跳过 | 用`df.describe(include='all')`或在dtypes里手动看字符串列 |
| loc和iloc取出来的行数不一样 | 混淆了标签和位置的右边界行为 | loc包含右边界(标签)、iloc不含右边界(位置)→回Cell 4b看坑4 |
| info()显示某列non-null<5000但不知道怎么办 | 发现了缺失但不知道怎么处理 | 今天先"发现并记录"→W4会学缺失值处理。现在用`df['col'].isna().sum()`量化 |

In [ ]:
## 🔙 回到开头的问题

# 还记得mentor小王的三个问题吗？
# ①哪些城市卖得最好？②有没有异常大额订单？③数据有没有缺失？
#
# 现在用今天学的pandas，8行代码解决——比W1的40行少5倍，快100倍

# 你的代码——用今天学的武器：


---

## 今天你学会了吗？

- [ ] **初级（L2：能用）**：不看笔记，独立完成基础练习1和2
  没过 → 回看概念1的cell 3b-c和概念2的cell 4b-c，重做基础练习

- [ ] **中级（L3：能讲）**：下面写一段200字以内的白话，讲给不懂技术的同事听：
  "pandas是什么？loc和iloc有什么区别？为什么布尔索引必须用&不能用and？"
  
  *你的回答（用自己的话写）：*
  
  
  
  没过 → 你写不出来说明停留在L2，回到概念1的cell 3b看第④段，概念2的cell 4b看第④段

- [ ] **高级（L4：能判断）**：迁移练习（滴滴出行）独立完成了吗？
  一份你从没见过的数据，你能自己判断该用六板斧的哪几板、该用loc还是iloc、该筛选什么吗？
  没过 → 补做迁移练习，不许查笔记

---

### 📋 概念掌握检查表（对照今日知识地图逐项检验）

- [ ] pandas世界观：能解释Series=带标签的list、DataFrame=dict of Series、pandas是W1手写代码的工业版
- [ ] 数据读入：能写`pd.read_csv()`含encoding/nrows/usecols/dtype/parse_dates参数、知道三个路径陷阱
- [ ] 初探六板斧：拿到新数据第一反应是跑`info()`、能解读describe的8个统计量、知道shape/columns.tolist()/dtypes/value_counts怎么用
- [ ] 行列操作：能手写loc按标签选和iloc按位置选、知道loc包含右边界iloc不含、选多列必须双括号
- [ ] 布尔索引：能手写&/|/isin/~组合条件、知道为什么必须用&不能用and、每个条件必须加括号
- [ ] 排序统计：能手写sort_values/nlargest/unique/nunique/value_counts
- [ ] 列操作：能手写新增列/assign/drop/rename
- [ ] **核心检查**：每个pandas操作你能说出它对应的W1 Python手写版吗？
  - `pd.read_csv()` = W1的`open()+csv.DictReader()+for+list.append()`
  - `df.describe()` = W1手写`sum/len/max/min+排序取中位数`
  - `df[df['amount']>500]` = W1的`[o for o in orders if o['amount']>500]`
  - `df.sort_values('amount')` = W1的`sorted(orders, key=lambda x:x['amount'])`
  - `df['新列'] = df['金额']*0.87` = W1的`for o in orders: o['新列']=o['金额']*0.87`

---

**明天**：今天你学会了"看到新数据第一反应是什么"——六板斧初探。明天你将能用一行代码回答"每个城市的销售额是多少"——比W1手写for循环快100倍。

**今天的初探六板斧 + 明天的groupby分组聚合 = 完整的数据分析第一步：先看清（今天）→再汇总（明天）**

---

### 🗺️ 今天的概念在本周知识网络中的位置

- **今天的初探六板斧** → 明天groupby分组聚合的输入（先看清数据→再分组汇总）
- **今天的布尔索引** → W1D2-3的for+if向量化版 → W4数据清洗的基础
- **今天的loc/iloc** → 本周每天都会用到 → D5的`.loc`是根治SettingWithCopyWarning的解法
- **今天的pandas世界观** → 整个W2-W16都在用。理解了"pandas=W1的工业版"→今后的每个新函数你都能追溯到W1的原型

> **本周路线**：D1初探→D2分组聚合→D3多表合并→D4变换+向量化→D5透视表+防坑→D6周项目→D7复习